# Advanced Forecasting Experiments

## Objective

Evaluate alternative forecasting strategies beyond direct sales prediction to better understand demand drivers and improve business decision-making.

## Experiments

### Experiment 1: Open Stores Only

Train forecasting models exclusively on operational stores to remove trivial zero-sales observations and focus on true demand prediction.

### Experiment 2: Customer Demand Forecasting

Predict customer footfall using historical demand, promotions, competition and calendar features.

### Experiment 3: Two-Stage Demand Intelligence Pipeline

Predict customer demand first and then use predicted customer counts to improve sales forecasting.

## Goals

* Understand the impact of store closures on forecasting performance.
* Evaluate whether customer demand can be accurately predicted.
* Compare direct sales forecasting against a customer-driven forecasting pipeline.
* Generate business insights that can support inventory planning and pricing decisions.


## Open Stores Only Forecasting

### Objective

Remove closed-store observations from the dataset and evaluate forecasting performance on active stores only.

### Motivation

Feature importance analysis revealed that store operational status was the dominant predictor of sales. Since closed stores always generate zero sales, the model can learn this pattern easily.

By restricting training and evaluation to open stores only, the forecasting task becomes more focused on actual customer demand and sales behavior.

### Expected Outcome

This experiment will provide a more realistic assessment of demand forecasting performance and reveal the importance of historical sales, promotions and seasonal patterns when stores are operational.


In [1]:
# Load model-ready dataset

import pandas as pd

df = pd.read_csv(
    "../data/processed/model_ready_data.csv"
)

print(df.shape)
df.head()

(985989, 42)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,SchoolHoliday,CompetitionDistance,Promo2,...,StateHoliday_c,Sales_Lag_7,Sales_Lag_14,Sales_Lag_28,Sales_RollingMean_7,Sales_RollingMean_14,Sales_RollingMean_28,Sales_RollingStd_7,Sales_RollingStd_14,Sales_RollingStd_28
0,1,2,2013-01-29,3725,522,1,0,0,1270.0,0,...,0,5720.0,3900.0,0.0,4533.142857,4170.500000,4121.285714,2080.017823,1899.595898,2075.631146
1,1,3,2013-01-30,4601,560,1,0,0,1270.0,0,...,0,5578.0,4008.0,5530.0,4248.142857,4158.000000,4254.321429,2026.274696,1902.086951,1914.845455
2,1,4,2013-01-31,4709,571,1,0,0,1270.0,0,...,0,5195.0,4044.0,4327.0,4108.571429,4200.357143,4221.142857,1951.681400,1905.090008,1899.913267
3,1,5,2013-02-01,5633,658,1,0,0,1270.0,0,...,0,5586.0,4127.0,4486.0,4039.142857,4247.857143,4234.785714,1914.889329,1909.177546,1902.071860
4,1,6,2013-02-02,5970,701,1,0,0,1270.0,0,...,0,5598.0,5182.0,4997.0,4045.857143,4355.428571,4275.750000,1921.288841,1943.954681,1919.949818


In [2]:
# Keep only open stores

open_df = df[df["Open"] == 1].copy()

print(open_df.shape)

open_df["Open"].value_counts()

(818845, 42)


Open
1    818845
Name: count, dtype: int64

In [3]:
split_date = "2015-06-01"

train_open = open_df[
    open_df["Date"] < split_date
].copy()

valid_open = open_df[
    open_df["Date"] >= split_date
].copy()

print(train_open.shape)
print(valid_open.shape)

(760234, 42)
(58611, 42)


## Feature Preparation

### Target Variable

* Sales

### Excluded Features

#### Sales

Target variable to be predicted.

#### Customers

Customer counts are not available at prediction time in a direct forecasting scenario.

#### Date

Temporal information has already been captured through engineered calendar features.

#### Open

All observations in this experiment correspond to operational stores (`Open = 1`), making this feature constant and non-informative.

### Objective

Create a realistic forecasting dataset focused exclusively on active-store demand prediction.


In [4]:
TARGET = "Sales"

drop_cols = [
    "Sales",
    "Customers",
    "Date",
    "Open"
]

X_train_open = train_open.drop(columns=drop_cols)
y_train_open = train_open[TARGET]

X_valid_open = valid_open.drop(columns=drop_cols)
y_valid_open = valid_open[TARGET]

In [5]:
print("X_train_open:", X_train_open.shape)
print("X_valid_open:", X_valid_open.shape)

print("y_train_open:", y_train_open.shape)
print("y_valid_open:", y_valid_open.shape)

X_train_open: (760234, 38)
X_valid_open: (58611, 38)
y_train_open: (760234,)
y_valid_open: (58611,)


In [6]:
from xgboost import XGBRegressor

xgb_open = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)


In [7]:
xgb_open.fit(X_train_open, y_train_open)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [8]:
open_preds = xgb_open.predict(X_valid_open)

In [9]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

In [10]:
open_mae = mean_absolute_error(
    y_valid_open,
    open_preds
)

open_rmse = np.sqrt(
    mean_squared_error(
        y_valid_open,
        open_preds
    )
)

open_r2 = r2_score(
    y_valid_open,
    open_preds
)

print("Open Store XGBoost Results")
print("-" * 35)
print(f"MAE  : {open_mae:.2f}")
print(f"RMSE : {open_rmse:.2f}")
print(f"R²   : {open_r2:.4f}")

Open Store XGBoost Results
-----------------------------------
MAE  : 621.49
RMSE : 881.53
R²   : 0.9197


The all-store model achieved better metrics because store closures created an easy-to-learn zero-sales pattern. However, when restricted to operational stores only, the model focused on actual demand forecasting, providing a more realistic assessment of retail demand prediction

## Experiment 2: Customer Demand Forecasting

## Customer Demand Forecasting

### Objective

Predict future customer footfall using historical demand patterns, promotional activities, competition information and calendar features.

### Motivation

Exploratory analysis revealed that customer counts exhibit the strongest relationship with sales. Therefore, accurately forecasting customer demand may provide an alternative pathway for sales forecasting.

### Business Value

Customer demand forecasts can support:

* Inventory planning
* Workforce scheduling
* Promotional strategy
* Revenue forecasting

### Goal

Evaluate whether customer demand can be predicted accurately enough to serve as an intermediate stage in a two-step forecasting pipeline.


In [11]:
# Customer demand forecasting

customer_drop_cols = [
    "Sales",
    "Customers",
    "Date",

    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Lag_28",

    "Sales_RollingMean_7",
    "Sales_RollingMean_14",
    "Sales_RollingMean_28",

    "Sales_RollingStd_7",
    "Sales_RollingStd_14",
    "Sales_RollingStd_28"
]



In [12]:
split_date = "2015-06-01"

train_df = open_df[
    open_df["Date"] < split_date
].copy()

valid_df = open_df[
    open_df["Date"] >= split_date
].copy()

print(train_df.shape)
print(valid_df.shape)

(760234, 42)
(58611, 42)


In [13]:
X_train_cust = train_df.drop(columns=customer_drop_cols)
y_train_cust = train_df["Customers"]

X_valid_cust = valid_df.drop(columns=customer_drop_cols)
y_valid_cust = valid_df["Customers"]

print(X_train_cust.shape)
print(X_valid_cust.shape)

(760234, 30)
(58611, 30)


In [14]:
from xgboost import XGBRegressor

cust_xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

In [15]:
cust_xgb.fit(
    X_train_cust,
    y_train_cust
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [17]:
cust_preds = cust_xgb.predict(
    X_valid_cust
)

In [18]:
cust_mae = mean_absolute_error(
    y_valid_cust,
    cust_preds
)

cust_rmse = np.sqrt(
    mean_squared_error(
        y_valid_cust,
        cust_preds
    )
)

cust_r2 = r2_score(
    y_valid_cust,
    cust_preds
)

print("Customer Forecast Results")
print("-" * 35)
print(f"MAE  : {cust_mae:.2f}")
print(f"RMSE : {cust_rmse:.2f}")
print(f"R²   : {cust_r2:.4f}")

Customer Forecast Results
-----------------------------------
MAE  : 101.37
RMSE : 135.13
R²   : 0.8810


In [ ]:
# Add predictions to validation dataframe
valid_open["Predicted_Customers"] = cust_preds

In [21]:
train_pipeline = train_open.copy()
valid_pipeline = valid_open.copy()

In [22]:
valid_pipeline["Predicted_Customers"] = cust_preds

## Sales Forecasting with Customer Demand

In [ ]:
#Training data doesnt have customer spredictions, so we can only evaluate on validation set
#Therefor using actual customers in training data and predicted customers in validation data

In [ ]:
sales_drop_cols = [
    "Sales",
    "Date",
    "Open"
]
#Customers is retained as a feature for sales prediction, but we will use actual customers in training data and predicted customers in validation data  

In [24]:
X_train_sales_actual = train_open.drop(
    columns=sales_drop_cols
)

y_train_sales_actual = train_open["Sales"]


In [35]:
X_valid_sales_actual = valid_open.drop(
    columns=sales_drop_cols
)
X_valid_sales_actual = X_valid_sales_actual.drop(
    columns=['Predicted_Customers']
)

y_valid_sales_actual = valid_open["Sales"]

In [36]:
print(X_train_sales_actual.shape)
print(X_valid_sales_actual.shape)

(760234, 39)
(58611, 39)


In [30]:
"Predicted_Customers" in train_open.columns

False

In [37]:
"Predicted_Customers" in valid_open.columns

True

In [38]:
sales_actual_xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

In [39]:
sales_actual_xgb.fit(
    X_train_sales_actual,
    y_train_sales_actual
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [40]:
sales_actual_preds = sales_actual_xgb.predict(
    X_valid_sales_actual
)

In [41]:
actual_mae = mean_absolute_error(
    y_valid_sales_actual,
    sales_actual_preds
)

actual_rmse = np.sqrt(
    mean_squared_error(
        y_valid_sales_actual,
        sales_actual_preds
    )
)

actual_r2 = r2_score(
    y_valid_sales_actual,
    sales_actual_preds
)

print("Sales + Actual Customers")
print("-"*35)
print(f"MAE  : {actual_mae:.2f}")
print(f"RMSE : {actual_rmse:.2f}")
print(f"R²   : {actual_r2:.4f}")

Sales + Actual Customers
-----------------------------------
MAE  : 453.21
RMSE : 638.99
R²   : 0.9578


## using prediction of customers in training data


In [42]:
train_pred_customers = cust_xgb.predict(X_train_cust)
valid_pred_customers = cust_xgb.predict(X_valid_cust)

In [43]:
train_pipeline = train_open.copy()
valid_pipeline = valid_open.copy()

train_pipeline["Predicted_Customers"] = train_pred_customers
valid_pipeline["Predicted_Customers"] = valid_pred_customers

In [44]:
sales_drop_cols = [
    "Sales",
    "Date",
    "Open",
    "Customers"
]

In [45]:
X_train_pipeline = train_pipeline.drop(
    columns=sales_drop_cols
)

X_valid_pipeline = valid_pipeline.drop(
    columns=sales_drop_cols
)

y_train_pipeline = train_pipeline["Sales"]
y_valid_pipeline = valid_pipeline["Sales"]

In [46]:
print(X_train_pipeline.shape)
print(X_valid_pipeline.shape)

(760234, 39)
(58611, 39)


In [47]:
print(
    "Predicted_Customers" in X_train_pipeline.columns
)

print(
    "Customers" in X_train_pipeline.columns
)

True
False


In [48]:
pipeline_xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)


In [49]:
pipeline_xgb.fit(
    X_train_pipeline,
    y_train_pipeline
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [50]:
pipeline_preds = pipeline_xgb.predict(
    X_valid_pipeline
)

In [51]:
pipeline_mae = mean_absolute_error(
    y_valid_pipeline,
    pipeline_preds
)

pipeline_rmse = np.sqrt(
    mean_squared_error(
        y_valid_pipeline,
        pipeline_preds
    )
)

pipeline_r2 = r2_score(
    y_valid_pipeline,
    pipeline_preds
)

print("Two-Stage Pipeline Results")
print("-"*35)
print(f"MAE  : {pipeline_mae:.2f}")
print(f"RMSE : {pipeline_rmse:.2f}")
print(f"R²   : {pipeline_r2:.4f}")

Two-Stage Pipeline Results
-----------------------------------
MAE  : 616.70
RMSE : 872.60
R²   : 0.9213
